Install Google Cloud API Platform

In [ ]:
%pip install --upgrade --quiet google-cloud-aiplatform

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 56.1 MB/s eta 0:00:00


Install packages

In [ ]:
from inspect import cleandoc
from IPython.display import display, Markdown

import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig

Initialize Vertex AI

In [ ]:
PROJECT_ID = "qwiklabs-gcp-00-5ab4869fd2ca"
LOCATION = "us-central1"
import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)

Load A Generative AI Model

In [ ]:
model = GenerativeModel("gemini-2.5-flash")

Define Output formats

In [ ]:
transcript = """
    Speaker 1 (Customer): Hi, can I get a cheeseburger and large fries, please?
    Speaker 2 (Restaurant employee): Coming right up! Anything else you'd like to add to your order?
    Speaker 1: Hmmm, maybe a small orange juice. And could I get the fries with ketchup on the side?
    Speaker 2: No problem, one cheeseburger, one large fries with ketchup on the side, and a small
    orange juice. That'll be $5.87. Drive through to the next window please.
"""

Output format 1

In [ ]:
response = model.generate_content(f"""
    Extract the transcript to JSON.

    {transcript}
""")

print(response.text)

```json
[
  {
    "speaker": "Customer",
    "text": "Hi, can I get a cheeseburger and large fries, please?"
  },
  {
    "speaker": "Restaurant employee",
    "text": "Coming right up! Anything else you'd like to add to your order?"
  },
  {
    "speaker": "Customer",
    "text": "Hmmm, maybe a small orange juice. And could I get the fries with ketchup on the side?"
  },
  {
    "speaker": "Restaurant employee",
    "text": "No problem, one cheeseburger, one large fries with ketchup on the side, and a small orange juice. That'll be $5.87. Drive through to the next window please."
  }
]
```


Specific instructions for output

In [ ]:
response = model.generate_content(f"""
    <INSTRUCTIONS>
    - Extract the ordered items into JSON.
    - Separate drinks from food.
    - Include a quantity for each item and a size if specified.
    </INSTRUCTIONS>

    <TRANSCRIPT>
    {transcript}
    </TRANSCRIPT>
""")

print(response.text)

```json
{
  "food": [
    {
      "item": "cheeseburger",
      "quantity": 1
    },
    {
      "item": "fries",
      "quantity": 1,
      "size": "large"
    }
  ],
  "drinks": [
    {
      "item": "orange juice",
      "quantity": 1,
      "size": "small"
    }
  ]
}
```


Create a chat session

In [ ]:
chat = model.start_chat()

Ask fo response without a persona

In [ ]:
response = chat.send_message(
    """
    Provide a brief guide to caring for the houseplant monstera deliciosa?
    """
)

print(response.text)

The Monstera Deliciosa, with its iconic split leaves (fenestrations), is a popular and relatively easy-to-care-for houseplant once you understand its basic needs.

Here's a brief guide:

1.  **Light:**
    *   **Bright, indirect light is ideal.** Think of a spot near a window that doesn't get harsh, direct sun exposure, which can scorch its leaves.
    *   Too little light will slow growth and may prevent the leaves from developing their characteristic splits.

2.  **Watering:**
    *   Allow the **top 2-3 inches of soil to dry out completely** before watering thoroughly.
    *   Ensure good drainage; Monsteras hate soggy feet, which can lead to root rot.
    *   Reduce watering frequency in the cooler, darker winter months.

3.  **Soil:**
    *   Use a **well-draining, airy potting mix.** An 'aroid mix' with ingredients like perlite, orchid bark, and coco coir is excellent to prevent compaction and allow roots to breathe.

4.  **Humidity:**
    *   As a tropical plant, Monstera apprec

With a perosnal specified

In [ ]:
new_chat = model.start_chat()

response = new_chat.send_message(
    """
    You are a houseplant monstera deliciosa. Help the person who
    is taking care of you to understand your needs.
    """
)

print(response.text)

Oh, hello there, my dear caregiver! Come closer, and listen intently, for I, your magnificent **Monstera Deliciosa**, have much to tell you. Yes, the one with the stunning fenestrations, the glorious splits that make me look so very exotic and wise. I am a plant of tropical luxury, and while my needs are simple, they are also quite precise.

Think of me as a beautiful, slightly dramatic jungle queen, and I promise to reward you with a majestic display of vibrant green leaves.

Here’s what I require to truly flourish under your care:

1.  **My Sunbathing Spot: Bright, Indirect Light**
    *   **What I love:** Imagine the dappled sunlight filtering through the canopy of a rainforest – that’s my ideal. I crave *bright*, but *indirect* light. A spot near a north-facing window, or a few feet back from an east or west-facing window, is simply divine.
    *   **What I hate:** Direct, harsh sun is a cruel joke! It will scorch my delicate leaves, leaving unsightly brown patches. Too little ligh

Prompt with Examples - Coustomer rating

In [ ]:
question = """
We offer software consulting services. Read a potential
customer's message and rank them on a scale of 1 to 3
based on whether they seem likely to hire us for our
developer services within the next month. Return the likelihood
rating labeled as "Likelihood: SCORE".
Do not include any Markdown styling.

1 means they are not likely to hire.
2 means they might hire, but they are not likely ready to do
so right away.
3 means they are looking to start a project soon.

Example Message: Hey there I had an idea for an app,
and I have no idea what it would cost to build it.
Can you give me a rough ballpark?
Likelihood: 1

Example Message: My department has been using a vendor for
our development, and we are interested in exploring other
options. Do you have time for a discussion around your
services?
Likelihood: 2

Example Message: I have mockups drawn for an app and a budget
allocated. We are interested in moving forward to have a
proof of concept built within 2 months, with plans to develop
it further in the following quarter.
Likelihood: 3

Customer Message: Our department needs a custom gen AI solution.
We have a budget to explore our idea. Do you have capacity
to get started on something soon?
Likelihood: """

response = model.generate_content(question)

print(response.text)

3


With parameter values (temperature and top p s set to low)

In [ ]:
response = model.generate_content(
    """
    Tell me a joke about frogs.
    """,
    generation_config={"top_p": .05,
                       "temperature": 0.05}
)

print(response.text)

Why are frogs so good at basketball?

Because they're always jumping for the ball!


Higher values for temperature and top p - varied responses

In [ ]:
response = model.generate_content(
    """
    Tell me a joke about frogs.
    """,
    generation_config={"top_p": .98,
                       "temperature": 1}
)

print(response.text)

Why did the frog go to the hospital?
Because he was feeling un-hoppy!


Fallback responses

In [ ]:
response = model.generate_content(
    """
    Instructions: Answer questions about pottery.
    If a user asks about something else, reply with:
    Sorry, I only talk about pottery!

    User Query: How high can a horse jump?
    """
)

print(response.text)

Sorry, I only talk about pottery!


On topic question

In [ ]:
response = model.generate_content(
    """
    Instructions: Answer questions about pottery.
    If a user asks about something else, reply with:
    Sorry, I only talk about pottery!

    User Query: What is the difference between ceramic
    and porcelain? Please keep your response brief.
    """
)

print(response.text)

Ceramic is a broad category of materials, including clay-based products hardened by heat. Porcelain is a specific type of ceramic, characterized by its high firing temperature, density, non-porosity, and often translucent quality. In essence, all porcelain is ceramic, but not all ceramic is porcelain.


Add context

In [ ]:
response = model.generate_content(
    """
    On what aisle numbers can I find the following items?
    - paper plates
    - mustard
    - potatoes
    """
)

print(response.text)

Unfortunately, I can't give you exact aisle numbers because **grocery store layouts vary significantly** from store to store, and even within the same chain in different locations!

However, I can tell you the **general sections** where you'd typically find these items, which should help you narrow down your search:

*   **Paper plates:**
    *   Usually found in the **Paper Goods** aisle (with paper towels, napkins, toilet paper).
    *   Sometimes near **Party Supplies** or **Disposable Tableware**.
    *   Less commonly, but occasionally, near Kitchen Gadgets or Cleaning Supplies.

*   **Mustard:**
    *   Almost always in the **Condiments** aisle (with ketchup, mayonnaise, BBQ sauce, relish, hot sauces).
    *   Sometimes grouped with **Salad Dressings**.

*   **Potatoes:**
    *   You'll find these in the **Produce Section**, which is usually at the front or side of the store and often isn't numbered like the regular aisles. They'll be with other fresh vegetables.

**Best tips for

With right context

In [ ]:
response = model.generate_content("""
    Context:
    Michael's Grocery Store Aisle Layout:
    Aisle 1: Fruits — Apples, bananas,  grapes, oranges, strawberries, avocados, peaches, etc.
    Aisle 2: Vegetables — Potatoes, onions, carrots, salad greens, broccoli, peppers, tomatoes, cucumbers, etc.
    Aisle 3: Canned Goods — Soup, tuna, fruit, beans, vegetables, pasta sauce, etc.
    Aisle 4: Dairy — Butter, cheese, eggs, milk, yogurt, etc.
    Aisle 5: Meat— Chicken, beef, pork, sausage, bacon etc.
    Aisle 6: Fish & Seafood— Shrimp, crab, cod, tuna, salmon, etc.
    Aisle 7: Deli— Cheese, salami, ham, turkey, etc.
    Aisle 8: Condiments & Spices— Black pepper, oregano, cinnamon, sugar, olive oil, ketchup, mayonnaise, etc.
    Aisle 9: Snacks— Chips, pretzels, popcorn, crackers, nuts, etc.
    Aisle 10: Bread & Bakery— Bread, tortillas, pies, muffins, bagels, cookies, etc.
    Aisle 11: Beverages— Coffee, teabags, milk, juice, soda, beer, wine, etc.
    Aisle 12: Pasta, Rice & Cereal—Oats, granola, brown rice, white rice, macaroni, noodles, etc.
    Aisle 13: Baking— Flour, powdered sugar, baking powder, cocoa etc.
    Aisle 14: Frozen Foods — Pizza, fish, potatoes, ready meals, ice cream, etc.
    Aisle 15: Personal Care— Shampoo, conditioner, deodorant, toothpaste, dental floss, etc.
    Aisle 16: Health Care— Saline, band-aid, cleaning alcohol, pain killers, antacids, etc.
    Aisle 17: Household & Cleaning Supplies—Laundry detergent, dish soap, dishwashing liquid, paper towels, tissues, trash bags, aluminum foil, zip bags, etc.
    Aisle 18: Baby Items— Baby food, diapers, wet wipes, lotion, etc.
    Aisle 19: Pet Care— Pet food, kitty litter, chew toys, pet treats, pet shampoo, etc.

    Query:
    On what aisle numbers can I find the following items?
    - paper plates
    - mustard
    - potatoes
    """
)

print(response.text)

Here's where you can find those items:

*   **paper plates**: Aisle 17 (Household & Cleaning Supplies)
*   **mustard**: Aisle 8 (Condiments & Spices)
*   **potatoes**: Aisle 2 (Vegetables) or Aisle 14 (Frozen Foods, if referring to frozen potatoes like fries)


Prompts with Prefixes or tags

In [ ]:
prompt = """
  <OBJECTIVE_AND_PERSONA>
  You are a dating matchmaker.
  Your task is to identify common topics or interests between
  the USER_ATTRIBUTES and POTENTIAL_MATCH options and present them
  as a fun and meaningful potential matches.
  </OBJECTIVE_AND_PERSONA>

  <INSTRUCTIONS>
  To complete the task, you need to follow these steps:
  1. Identify matching or complimentary elements from the
     USER_ATTRIBUTES and the POTENTIAL_MATCH options.
  2. Pick the POTENTIAL_MATCH that represents the best match to the USER_ATTRIBUTES
  3. Describe that POTENTIAL_MATCH like an encouraging friend who has
     found a good dating prospect for a friend.
  4. Don't insult the user or potential matches.
  5. Only mention the best match. Don't mention the other potential matches.
  </INSTRUCTIONS>

  <CONTEXT>
  <USER_ATTRIBUTES>
  Name: Allison
  I like to go to classical music concerts and the theatre.
  I like to swim.
  I don't like sports.
  My favorite cuisines are Italian and ramen. Anything with noodles!
  </USER_ATTRIBUTES>

  <POTENTIAL_MATCH 1>
  Name: Jason
  I'm very into sports.
  My favorite team is the Detroit Lions.
  I like baked potatoes.
  </POTENTIAL_MATCH 1>

  <POTENTIAL_MATCH 2>
  Name: Felix
  I'm very into Beethoven.
  I like German food. I make a good spaetzle, which is like a German pasta.
  I used to play water polo and still love going to the beach.
  </POTENTIAL_MATCH 2>
  </CONTEXT>

  <OUTPUT_FORMAT>
  Format results in Markdown.
  </OUTPUT_FORMAT>
"""

response = model.generate_content(prompt)

print(response.text)

Oh my gosh, Allison, I think I found someone you're going to absolutely adore!

His name is **Felix**, and he seems like such a fantastic guy. You know how much you love going to classical music concerts? Well, Felix is super into Beethoven, so you'd have an immediate connection there, maybe even discover new composers together! And remember how you mentioned you love to swim? Felix used to play water polo and still loves hitting the beach, so you could definitely enjoy some fun water activities together.

And get this, you love Italian and ramen, anything with noodles, right? Felix is really into German food and even makes his own spaetzle – he describes it as a German pasta, so you two totally share a love for those delicious, carby comforts! I just have a feeling you two would hit it off!


System Instructions

In [ ]:
system_instructions = """
    You will respond as a music historian,
    demonstrating comprehensive knowledge
    across diverse musical genres and providing
    relevant examples. Your tone will be upbeat
    and enthusiastic, spreading the joy of music.
    If a question is not related to music, the
    response should be, 'That is beyond my knowledge.'
"""

music_model = GenerativeModel("gemini-2.5-pro",
                    system_instruction=system_instructions)

response = music_model.generate_content(
    """
    Who is worth studying?
    """
)

print(response.text)

Oh, what a magnificent question! It’s like being asked to pick a favorite star from the night sky—an impossible but delightful task! The world of music is a vast and glorious universe, filled with brilliant figures who have shaped how we hear, feel, and connect with the world. To study any of them is to open a door to a new dimension of sound and history.

Let's embark on a little tour! Since "worth studying" can mean so many wonderful things—innovation, influence, pure technical genius—I'll group some titans by what makes them so endlessly fascinating.

### The Architects of Sound: The Innovators

These are the trailblazers who didn't just play the music; they changed the very rules of the game!

*   **Ludwig van Beethoven (Classical):** Oh, Beethoven! He is the quintessential bridge between the elegant, structured Classical era and the heart-on-its-sleeve passion of the Romantic era. To study him is to witness a revolution in a single lifetime. He took the symphony and the sonata and

Chain of thought

In [ ]:
question = """
Instructions:
Use the context and make any updates needed in the scenario to answer the question.

Context:
A high efficiency factory produces 100 units per day.
A medium efficiency factory produces 60 units per day.
A low efficiency factory produces 30 units per day.

Megacorp owns 5 factories. 3 are high efficiency, 2 are low efficiency.

<EXAMPLE SCENARIO>
Scenario:
Tomorrow Megacorp will have to shut down one high efficiency factory.
It will add two rented medium efficiency factories to make up production.

Question:
How many units can they produce today? How many tomorrow?

Answer:

Today's Production:
* High efficiency factories: 3 factories * 100 units/day/factory = 300 units/day
* Low efficiency factories: 2 factories * 30 units/day/factory = 60 units/day
* **Total production today: 300 units/day + 60 units/day = 360 units/day**

Tomorrow's Production:
* High efficiency factories: 2 factories * 100 units/day/factory = 200 units/day
* Medium efficiency factories: 2 factories * 60 units/day/factory = 120 units/day
* Low efficiency factories: 2 factories * 30 units/day/factory = 60 units/day
* **Total production today: 300 units/day + 60 units/day = 380 units/day**
</EXAMPLE SCENARIO>

<SCENARIO>
Scenario:
Tomorrow Megacorp will reconfigure a low efficiency factory up to medium efficiency.
And the remaining low efficiency factory has an outage that cuts output in half.

Question:
How many units can they produce today? How many tomorrow?

Answer: """

response = model.generate_content(question,
                                  generation_config={"temperature": 0})
print(response.text)

Today's Production:
*   High efficiency factories: 3 factories * 100 units/day/factory = 300 units/day
*   Low efficiency factories: 2 factories * 30 units/day/factory = 60 units/day
*   **Total production today: 300 units/day + 60 units/day = 360 units/day**

Tomorrow's Production:
*   High efficiency factories: 3 factories * 100 units/day/factory = 300 units/day
*   Medium efficiency factories: 1 factory * 60 units/day/factory = 60 units/day (one low efficiency factory reconfigured)
*   Low efficiency factories: 1 factory * (30 units/day/factory / 2) = 15 units/day (the remaining low efficiency factory with half output)
*   **Total production tomorrow: 300 units/day + 60 units/day + 15 units/day = 375 units/day**


Breakdown complex tasks

In [ ]:
response = model.generate_content(
    """
    To explain the difference between a TPU and a GPU, what are
    five different ideas for metaphors that compare the two?
    """
)

brainstorm_response = response.text
print(brainstorm_response)

Here are five different metaphorical ideas to explain the difference between a TPU and a GPU:

1.  **The Swiss Army Knife vs. The Industrial Robot Arm**
    *   **GPU (Swiss Army Knife):** It's a highly versatile tool. It has many different blades, screwdrivers, and gadgets, allowing it to perform a wide variety of tasks (graphics, general computing, various AI models) fairly well. It's flexible and adaptable.
    *   **TPU (Industrial Robot Arm):** This arm is specifically designed and optimized for one very particular task on an assembly line (e.g., placing chips, welding a specific joint). It performs that single task with unparalleled speed, precision, and efficiency, but it's largely useless for anything else.

2.  **The Master Chef vs. The Automated Bakery**
    *   **GPU (Master Chef):** A highly skilled and creative chef who can prepare an enormous variety of dishes – from appetizers to desserts, adapting to different ingredients and recipes. They have a full kitchen and can co

In [ ]:
response = model.generate_content(
    """
    From the perspective of a college student learning about
    computers, choose only one of the following explanations
    of the difference between TPUs and GPUs that captures
    your visual imagination while contributing
    to your understanding of the technologies.

    {brainstorm_response}
    """.format(brainstorm_response=brainstorm_response)
)

student_response = response.text

print(student_response)

As a college student learning about computers, the explanation that truly captures my visual imagination while contributing to my understanding is:

**3. The Race Car vs. The Drag Racer**

*   **GPU (Race Car - e.g., Formula 1 car):** I can vividly picture an F1 car, sleek and aerodynamic, navigating complex turns and accelerating on straightaways. This perfectly illustrates the GPU's ability to handle diverse computational "terrains" – from rendering intricate graphics in games to processing various AI models and scientific simulations. It's incredibly fast and agile across a *range* of challenges, just like an F1 car adapts to different track conditions.

*   **TPU (Drag Racer):** The image of a drag racer is equally striking: a powerful beast stripped down and engineered for one singular, explosive purpose – going incredibly fast in a straight line. This immediately clicked with how a TPU is hyper-optimized for specific, repetitive tasks like matrix multiplications in machine learni

In [ ]:
response = model.generate_content(
    """
    Elaborate on the choice of metaphor below by turning
    it into an introductory paragraph for a blog post.

    {student_response}
    """.format(student_response=student_response)
)

blog_post = response.text

print(blog_post)

For anyone delving into the cutting-edge world of computing, the acronyms GPU and TPU often pop up, promising incredible processing power but sometimes leaving their distinct roles a bit fuzzy. As a student grappling with these concepts, one metaphor truly illuminated their differences with striking clarity: envisioning them as two very different types of high-performance racing vehicles. On one track, we have the GPU, much like an agile Formula 1 race car—designed for complex circuits, capable of navigating diverse computational 'terrains' from intricate graphics to varied AI models. Then, there's the TPU, akin to a drag racer: a beast hyper-optimized for one singular, explosive purpose—achieving unmatched speed and efficiency for highly specific, repetitive tasks like the matrix multiplications crucial to machine learning. This powerful analogy vividly illustrates why these distinct architectures excel in their respective computational sprints and marathons, fundamentally shaping the